---
title: Word Game Hacks/Changes
description: Explore my changes in the Word Game and how I created the new Features
comments: false
layout: post
permalink: /Word_Game/lesson
---

### MAJOR FEATURE: Persistent Leaderboard System
Description
A comprehensive leaderboard system that tracks and displays the top 5 scores across all game sessions. Scores are calculated using a formula that considers WPM, accuracy, and difficulty multipliers to create fair competition across different game modes.

### Leaderboard SubFeatures

The leaderboard system enhances the competitive aspect of the typing game by:

1. Persistent Score Tracking: Maintains top scores across game sessions
2. Difficulty-Based Scoring: Uses multipliers to balance different difficulty levels
3. Comprehensive Metrics: Considers both WPM and accuracy for fair scoring
4. Visual Ranking Display: Shows top 5 performers with difficulty indicators

Scoring Formula: Score = WPM × Difficulty_Multiplier × (Accuracy/100)

This system encourages players to:
- Improve their typing speed and accuracy
- Challenge themselves with higher difficulties for better scores
- Compete against their previous performances
- Track progress over time

The leaderboard updates automatically after each completed game and displays:
- Player rank (1-5)
- WPM achieved
- Accuracy percentage  
- Difficulty level symbol

In [ ]:
%%javascript
// Leaderboard data structure and storage
function loadLeaderboard() {
    // If a global leaderboard array doesn't exist yet, create an empty one.
    // We use a global variable because localStorage is unavailable in this environment.
    if (!window.gameLeaderboard) {
        window.gameLeaderboard = [];
    }
    // Refresh the on-screen leaderboard with the current (possibly empty) data.
    updateLeaderboardDisplay();
}

// Score calculation with difficulty multipliers
function saveScore(wpm, accuracy, difficulty, stringType) {
    // Create a new score object containing all relevant data for the current game.
    const score = {
        // Convert words per minute input to an integer.
        wpm: parseInt(wpm),
        // Remove the '%' sign from accuracy and convert it to an integer.
        accuracy: parseInt(accuracy.replace('%', '')),
        // Store the difficulty setting (e.g., easy, medium, hard).
        difficulty: difficulty,
        // Store the type of string or text the user typed.
        stringType: stringType,
        // Compute an overall score using WPM, difficulty multiplier, and accuracy percentage.
        score: Math.round(
            parseInt(wpm) *
            difficulties[difficulty].multiplier *
            (parseInt(accuracy.replace('%', '')) / 100)
        ),
        // Record the current date and time for reference.
        timestamp: new Date().toLocaleString()
    };

    // Ensure the global leaderboard array exists before adding the new score.
    if (!window.gameLeaderboard) {
        window.gameLeaderboard = [];
    }
    
    // Add the new score entry to the leaderboard.
    window.gameLeaderboard.push(score);
    // Sort scores in descending order so highest score appears first.
    window.gameLeaderboard.sort((a, b) => b.score - a.score);
    // Keep only the top 5 scores to limit leaderboard size.
    window.gameLeaderboard = window.gameLeaderboard.slice(0, 5);
    // Update the displayed leaderboard so users immediately see the new ranking.
    updateLeaderboardDisplay();
}

// Dynamic leaderboard display updates
function updateLeaderboardDisplay() {
    // If there are no scores yet, show a placeholder message.
    if (!window.gameLeaderboard || window.gameLeaderboard.length === 0) {
        leaderboardContent.innerHTML =
            '<div class="leaderboard-entry"><span>No scores yet</span><span>--</span></div>';
        return;
    }

    // Clear any existing leaderboard HTML so we can rebuild it fresh.
    leaderboardContent.innerHTML = '';
    // Loop through each score entry and create a visual row.
    window.gameLeaderboard.forEach((entry, index) => {
        // Create a new div element to hold a single leaderboard entry.
        const div = document.createElement('div');
        div.className = 'leaderboard-entry';
        // Get a symbol (like ★ or ☠) to represent the difficulty for this entry.
        const diffSymbol = difficulties[entry.difficulty].symbol;
        // Fill the div with rank number, WPM, difficulty symbol, and accuracy percentage.
        div.innerHTML = `
            <span>${index + 1}. ${entry.wpm} WPM ${diffSymbol}</span>
            <span>${entry.accuracy}%</span>
        `;
        // Append the newly created entry div to the main leaderboard container.
        leaderboardContent.appendChild(div);
    });
}


<IPython.core.display.Javascript object>

/* Leaderboard styling */
.leaderboard {
    background: rgba(0, 0, 0, 0.7);
    border-radius: 15px;
    padding: 20px;
    margin: 20px auto;
    width: 300px;
    text-align: center;
    box-shadow: 0 8px 25px rgba(0, 0, 0, 0.3);
}

.leaderboard h3 {
    color: #FFD700;
    margin-bottom: 15px;
    text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.8);
}

.leaderboard-entry {
    display: flex;
    justify-content: space-between;
    padding: 8px 0;
    border-bottom: 1px solid rgba(255, 255, 255, 0.1);
    font-size: 14px;
}

In [ ]:
<!-- HTML structure for leaderboard -->
<div class="leaderboard">
    <h3>🏆 Top Scores</h3>
    <div id="leaderboardContent">
        <div class="leaderboard-entry">
            <span>No scores yet</span>
            <span>--</span>
        </div>
    </div>
</div>

### MINI FEATURE 1: Current Character Highlighting
Description
Visual indicator that highlights the next character to be typed with a yellow background, making it easier for users to see exactly where they are in the text and what character comes next.

### Mini Feature #1 SubFeatures
The current character highlighting system improves user experience by:

1. Visual Guidance: Shows exactly which character to type next
2. Enhanced Focus: Reduces eye strain by clearly indicating current position
3. Better Accuracy: Helps users avoid looking ahead and making mistakes
4. Real-time Updates: Moves dynamically as user types each character

Implementation Details:
- Uses a semi-transparent yellow background highlight
- Positioned precisely behind the current character
- Updates in real-time with each keystroke
- Maintains visibility across different text layouts and line wrapping

This feature particularly helps:
- New typists who need guidance on positioning
- Users practicing accuracy over speed  
- Anyone working with complex text patterns
- Reducing cognitive load during typing sessions

In [ ]:
%%javascript
// Enhanced drawUserText function with character highlighting
function drawUserText(prompt, input) {
    // Clear the entire canvas so we can redraw fresh text every frame.
    wordCtx.clearRect(0, 0, wordCanvas.width, wordCanvas.height);
    // Set the font style for the text to be drawn.
    wordCtx.font = '24px Arial';
    // Align text drawing from the left edge (used when measuring text positions).
    wordCtx.textAlign = 'left';

    // Determine the maximum width of each text line so words can wrap correctly.
    const maxWidth = wordCanvas.width - 20;
    // Vertical spacing between consecutive text lines.
    const lineHeight = 30;
    // Split the prompt string into multiple lines so it fits the canvas width.
    const lines = wrapText(prompt, maxWidth);
    // Calculate a starting Y position so the entire block of text is vertically centered.
    const startY = (wordCanvas.height - lines.length * lineHeight) / 2;

    // Draw each wrapped line individually.
    lines.forEach((line, lineIndex) => {
        // Y coordinate of this specific line.
        const lineY = startY + lineIndex * lineHeight;
        // Center this line horizontally by subtracting its measured width from canvas width.
        const lineX = (wordCanvas.width - wordCtx.measureText(line).width) / 2;
        
        // Track the current X position as we draw character by character.
        let currentX = lineX;
        // Determine the absolute character index in the full prompt where this line starts.
        // Add 1 space character for each line break except the first.
        const startCharIndex = lines.slice(0, lineIndex).join(' ').length + (lineIndex > 0 ? 1 : 0);
        // Determine the absolute character index where this line ends.
        const endCharIndex = startCharIndex + line.length;

        // Draw each character individually for precise highlighting and color changes.
        for (let i = startCharIndex; i < endCharIndex; i++) {
            // The actual character from the prompt at this index.
            const char = prompt[i] || '';
            // Default color for untyped characters (a light gray).
            let color = '#dededeff';
            
            if (i < input.length) {
                // The user has typed something for this character position.
                const typedChar = input[i];
                // If the typed character matches the prompt, color it green, else red.
                color = typedChar === char ? '#00ff00' : '#ff0000';
            } else if (i === input.length) {
                // The current character the user is about to type.
                // Draw a translucent yellow rectangle to highlight the next character position.
                wordCtx.fillStyle = 'rgba(255, 255, 0, 0.4)';
                wordCtx.fillRect(
                    currentX - 2,           // Slightly left of the character start for padding.
                    lineY - 22,             // Position rectangle slightly above the text baseline.
                    wordCtx.measureText(char).width + 4, // Width slightly larger than the character.
                    26                      // Height to cover the text fully.
                );
                // Make the current character itself bright white for emphasis.
                color = '#ffffff';
            }
            
            // Set the fill color for this specific character.
            wordCtx.fillStyle = color;
            // Draw the character at the current X position along the baseline of this line.
            wordCtx.fillText(char, currentX, lineY);
            // Advance currentX by the width of this character so the next character lines up correctly.
            currentX += wordCtx.measureText(char).width;
        }
    });
}


<IPython.core.display.Javascript object>

### MINI FEATURE 2: Advanced Difficulty System
Description
A multi-tier difficulty system that progressively increases challenge through different typing constraints and provides score multipliers to maintain fair competition across difficulty levels.

The difficulty system adds depth and replayability through four distinct modes:

1. Normal Mode (1x multiplier):
   - Standard typing with backspace allowed
   - Case insensitive matching
   - Most forgiving for beginners

2. Hard Mode (1.5x multiplier):  
   - No backspace allowed - mistakes must be overtyped
   - Case insensitive matching
   - Teaches accuracy under pressure

3. Expert Mode (2x multiplier):
   - No backspace allowed
   - Case sensitive matching  
   - Requires precise typing skills

4. Insane Mode (3x multiplier):
   - No backspace allowed
   - Case sensitive matching
   - Game ends immediately on first mistake
   - Ultimate challenge for expert typists

The difficulty system provides:
- Progressive skill development pathway
- Score multipliers to maintain fair competition
- Different typing constraints to improve various skills
- Visual indicators to show current difficulty level

Each mode teaches different typing skills:
- Normal: Basic speed and familiarity
- Hard: Accuracy under pressure (no corrections)
- Expert: Precision typing with case sensitivity
- Insane: Perfect execution under maximum constraints

In [ ]:
%%javascript
// Difficulty configuration system
const difficulties = {
    // "Normal" mode allows standard typing with backspace enabled.
    normal: { 
        name: 'Normal', 
        color: 'easy',          // CSS class for color styling
        symbol: '●',            // Symbol used in UI to represent this difficulty
        multiplier: 1,          // Scoring multiplier (base level)
        description: 'Standard typing with backspace allowed'
    },
    // "Hard" mode disables backspace—mistakes must be overtyped.
    hard: { 
        name: 'Hard', 
        color: 'medium',
        symbol: '◆',
        multiplier: 1.5,        // Slightly higher score multiplier to reward challenge
        description: 'No backspace - mistakes must be overtyped'
    },
    // "Expert" mode disables backspace and enforces case sensitivity.
    expert: { 
        name: 'Expert', 
        color: 'hard',
        symbol: '★',
        multiplier: 2,          // Higher multiplier due to increased difficulty
        description: 'No backspace + case sensitive'
    },
    // "Insane" mode requires perfection: no backspace, case sensitive, and any mistake ends the game.
    insane: { 
        name: 'Insane', 
        color: 'expert',
        symbol: '⚡',
        multiplier: 3,          // Highest multiplier to reflect extreme challenge
        description: 'No backspace + case sensitive + no mistakes allowed'
    }
};

// Update the UI to show the currently selected difficulty.
function updateDifficultyDisplay() {
    // Retrieve the current difficulty object from the difficulties map.
    const diff = difficulties[currentDifficulty];
    // Update the button text to show the current difficulty name.
    difficultyToggle.textContent = `Difficulty: ${diff.name}`;
    // Display the difficulty's symbol, name, and description with color styling.
    difficultyIndicator.innerHTML = `<span class="${diff.color}">${diff.symbol}</span> ${diff.name} - ${diff.description}`;
}

// When the difficulty toggle button is clicked, cycle through available difficulties.
difficultyToggle.addEventListener('click', () => {
    // Get an array of the difficulty keys: ['normal','hard','expert','insane'].
    const diffKeys = Object.keys(difficulties);
    // Find the index of the current difficulty in that array.
    const currentIndex = diffKeys.indexOf(currentDifficulty);
    // Move to the next difficulty, wrapping around to the first when reaching the end.
    currentDifficulty = diffKeys[(currentIndex + 1) % diffKeys.length];
    // Refresh the on-screen display to reflect the new difficulty.
    updateDifficultyDisplay();
});

// Handle all keyboard input and enforce difficulty-based rules.
document.onkeydown = function (e) {
    // If the game is finished, ignore further input.
    if (finished) return;

    // If a single character key is pressed and we haven't reached the end of the string:
    if (e.key.length === 1 && userInput.length < selectedString.length) {
        // Get the correct character at the current position.
        const nextChar = selectedString[userInput.length];
        let isCorrect = false;
        
        // Apply difficulty rules for case sensitivity.
        if (currentDifficulty === 'expert' || currentDifficulty === 'insane') {
            // Case-sensitive comparison.
            isCorrect = e.key === nextChar;
        } else {
            // Case-insensitive comparison for normal and hard modes.
            isCorrect = e.key.toLowerCase() === nextChar.toLowerCase();
        }
        
        // In "Insane" mode, a single incorrect character ends the game immediately.
        if (currentDifficulty === 'insane' && !isCorrect) {
            alert('Game Over! Insane mode requires perfect accuracy.');
            return;
        }
        
        // If the character was incorrect (for modes that allow mistakes), increment mistake count.
        if (!isCorrect) {
            mistakes++;
        }
        
        // Append the typed character to the user's input string.
        userInput += e.key;
    } else if (e.key === 'Backspace' && userInput.length > 0) {
        // Handle backspace depending on difficulty.
        if (currentDifficulty === 'normal') {
            // Only normal mode allows backspacing to delete the last character.
            userInput = userInput.slice(0, -1);
        }
        // Hard, Expert, and Insane modes block backspace entirely.
    }

    // Redraw the user text on the canvas to reflect the latest input and highlights.
    drawUserText(selectedString, userInput);
    // Update stats such as WPM, accuracy, etc., based on the new input state.
    updateStats(selectedString, userInput, startTime);

    // If the entire string has been correctly typed, trigger the end-of-game sequence.
    if (userInput === selectedString) {
        finishGame(selectedString, userInput, startTime);
    }
};


<IPython.core.display.Javascript object>

In [ ]:
<!-- HTML structure for difficulty system -->
<button id="difficultyToggle">Difficulty: Normal</button>

<div style="text-align: center;">
    <div class="difficulty-indicator" id="difficultyIndicator">
        <span class="easy">●</span> Normal Mode
    </div>
</div>

### Summary of Changes/Hacks

MAJOR FEATURE - Persistent Leaderboard:
Tracks top 5 scores across sessions
Difficulty-based scoring system
Visual feedback on updates
Rank display in completion alerts

MINI FEATURE 1 - Current Character Highlighting:
Yellow background highlight on next character
Real-time position tracking
Enhanced typing guidance
Cross-platform visual consistency

MINI FEATURE 2 - Advanced Difficulty System:
Four progressive difficulty modes
Backspace restrictions by mode
Case sensitivity options
Score multipliers for fair competition

PRESERVED ORIGINAL FEATURES:
Real-time WPM calculation
Real-time accuracy tracking  
Text wrapping and display
Options menu functionality
Mistake tracking system
Canvas-based rendering

### Key Programming Concepts Used

Event listeners for user input capture
Canvas API for dynamic graphics rendering
Array manipulation for text wrapping and leaderboard sorting
Real-time calculations for performance metrics
Object-oriented data structures for game state management
Conditional logic for difficulty-based game rules
Mathematical operations for displayed stats using formulas and calculations

### How is Canvas Used?

Canvas Text Rendering

drawText() - Displays initial prompt text centered on canvas
drawUserText() - Shows typed text with color coding (green=correct, red=wrong)
wrapText() - Handles line breaks for long text strings

Canvas Enhanced Features

In [ ]:
%%javascript
// Character-by-character rendering for precise control
for (let i = startCharIndex; i < endCharIndex; i++) {
    // Grab the character from the full prompt at the absolute index i.
    const char = prompt[i];
    
    // Color coding based on typing accuracy:
    // If the user has already typed at this position, choose green for correct and red for incorrect.
    if (i < input.length) {
        // Ternary keeps this concise: green if typed character equals prompt char, else red.
        color = (input[i] === char) ? '#00ff00' : '#ff0000';
    }
    // Current character highlighting (Mini Feature 1):
    // If this index is exactly where the user is expected to type next (caret position),
    // draw a translucent yellow rectangle behind the character and set the char color to white.
    else if (i === input.length) {
        // Set a translucent yellow fill for the highlight rectangle.
        wordCtx.fillStyle = 'rgba(255, 255, 0, 0.4)';
        // Draw the rectangle slightly padded horizontally (currentX-2 to width+4) and vertically
        // such that it covers the glyph area. Uses charWidth as the approximate glyph width.
        wordCtx.fillRect(currentX - 2, lineY - 22, charWidth + 4, 26);
        // Make the current character itself bright white so it stands out on the highlight.
        color = '#ffffff';
    }
    
    // Set the fill color (either green/red/white/default previously defined) and draw the character.
    wordCtx.fillStyle = color;
    wordCtx.fillText(char, currentX, lineY);
    // Advance the drawing X position by the exact measured width of this character so spacing is accurate.
    currentX += wordCtx.measureText(char).width;
}


<IPython.core.display.Javascript object>

### Canvas Coordinate System Usage

Character positioning: measureText() calculates exact character widths
Line wrapping: Multi-line text positioned using calculated lineY coordinates
Dynamic highlighting: fillRect() draws background rectangles behind specific characters
Real-time updates: clearRect() clears canvas before redrawing each frame

### Canvas API Methods Used

clearRect() - Erases canvas before each redraw
fillText() - Renders individual characters with precise positioning
fillRect() - Draws highlight rectangles behind characters
measureText() - Calculates character widths for positioning
fillStyle - Sets colors for text and highlighting dynamically